# rlmflow node basics

A tour of the `Node` query API: walk a saved run, find specific agents, filter by type / depth, compare snapshots, and pull message history out of `Session`. Pure Python — no rendering, no LLM calls.

The trace under `examples/data/notebook-coding-agent/` was generated by [`coding_agent.ipynb`](./coding_agent.ipynb). For visualizations on top of the same trace, see [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb).

**What we cover**

1. Loading a trace
2. Walking the subtree — `state.walk()`
3. Finding a specific node — `state.find`, `state.path_to`
4. Filtering — `state.where`, `state.leaves`, `state.results`, `state.errors`
5. Diffing two snapshots — `state.diff`
6. Reading the session — `session.load`, `session.chain_to`
7. Streaming events to sinks — `tee`, `json_logs`


## 1. Load a trace

A saved run is a list of `Node` snapshots — one complete tree per engine tick. We pick the *richest* snapshot (most nodes) so queries against the final structure return everything; early snapshots only show what had been produced by that point.

In [1]:
from rlmflow.utils.trace import load_trace

trace = load_trace("../data/notebook-coding-agent/trace")
states = trace.states
richest = max(states, key=lambda s: len(s.walk()))
print(f"loaded {len(states)} states  ·  richest has {len(richest.walk())} nodes")

loaded 4 states  ·  richest has 7 nodes


## 2. Walking the subtree

`state.walk()` returns every node in DFS order. Useful for counting, scanning, building summaries.

In [2]:
print("all agents in the subtree:")
for n in richest.walk():
    print(f"  {n.agent_id:<20} [{n.type}] depth={n.depth}")

all agents in the subtree:
  root                 [supervising] depth=0
  root.index_html      [query] depth=1
  root.styles_css      [query] depth=1
  root.boid_js         [query] depth=1
  root.simulation_js   [query] depth=1
  root.renderer_js     [query] depth=1
  root.main_js         [query] depth=1


## 3. Finding a specific node

`find(target)` accepts either a `node.id` (exact) or an `agent_id` (first match). `path_to(target)` returns the root-to-target ancestor chain.

In [3]:
target = "root.index_html"
hit = richest.find(target)
print(hit.agent_id, hit.type, "->", str(getattr(hit, "result", ""))[:60])

print(f"\npath_to({target!r}):")
for n in richest.path_to(target):
    print(f"  {n.agent_id} [{n.type}]")

root.index_html query -> 

path_to('root.index_html'):
  root [supervising]
  root.index_html [query]


## 4. Filtering

Convenience selectors: `leaves()` (no children), `results()` (`ResultNode`s), `errors()` (`ErrorNode`s). `where(...)` is the general case — keyword args are attribute matches, or pass a callable predicate.

In [4]:
print("leaves :", [n.agent_id for n in richest.leaves()])
print("results:", [n.agent_id for n in richest.results()])
print("errors :", richest.errors() or "(none)")

leaves : ['root.index_html', 'root.styles_css', 'root.boid_js', 'root.simulation_js', 'root.renderer_js', 'root.main_js']
results: []
errors : (none)


In [5]:
# kwargs match attributes
deep_results = richest.where(type="result", depth=1)
print("via kwargs    :", [n.agent_id for n in deep_results])

# or pass a predicate
same = richest.where(lambda n: n.depth >= 1 and n.type == "result")
print("via predicate :", [n.agent_id for n in same])

via kwargs    : []
via predicate : []


## 5. Diffing two snapshots

`state.diff(other)` returns `(added, removed)` indexed by node id — what appeared and disappeared between two snapshots. Handy for understanding what each step actually contributed.

In [6]:
early = states[0]
late  = states[-1]
diff  = late.diff(early)
print(f"{len(diff.added)} nodes added between step 0 and step {len(states) - 1}")
for n in diff.added[:6]:
    print(f"  + {n.agent_id} [{n.type}]")

7 nodes added between step 0 and step 3
  + root [result]
  + root.index_html [result]
  + root.styles_css [result]
  + root.boid_js [result]
  + root.simulation_js [result]
  + root.renderer_js [result]


## 6. Reading the session

`Workspace.session` persists the full node/message graph. `session.load()` returns `{node_id: Node}`; `session.chain_to(node)` reconstructs the message chain that produced a given node — what the LLM actually saw on its last turn.

In [7]:
from pathlib import Path
from rlmflow.workspace import FileSession

session = FileSession(Path("../data/notebook-coding-agent/session"))
all_nodes = session.load()
print(f"{len(all_nodes)} nodes persisted across {len({n.agent_id for n in all_nodes.values()})} agents")

22 nodes persisted across 7 agents


In [8]:
# Reconstruct the full chain ending at the terminal node of `root.index_html`.
agent_nodes = [n for n in all_nodes.values() if n.agent_id == "root.index_html"]
priority = {"result": 0, "error": 1, "observation": 2, "action": 3, "supervising": 4, "query": 5}
leaf = min(agent_nodes, key=lambda n: priority.get(n.type, 99))
chain = session.chain_to(leaf)
for n in chain:
    print(f"  {n.type:<12} {n.agent_id}")

  query        root.index_html
  action       root.index_html
  result       root.index_html


## 7. Streaming events to sinks

`tee(stream, *sinks)` fans a step iterator out to multiple callables — handy during a live run (write to a file, post to Slack, push to a UI, etc.). `json_logs(states, path)` writes a newline-delimited JSON event log for external tools (Loki, Datadog, DuckDB).

In [9]:
from rlmflow.utils.viz import tee

events = []
for _ in tee(richest.walk(), events.append):
    pass
print(f"tee forwarded {len(events)} nodes to a sink")

tee forwarded 7 nodes to a sink


In [10]:
import tempfile
from rlmflow.utils.tracing import json_logs

with tempfile.NamedTemporaryFile(suffix=".jsonl", delete=False) as f:
    out = json_logs(states, f.name)
lines = out.read_text().splitlines()
print(f"json_logs wrote {len(lines)} lines to {out}")
print("first line preview:")
print(lines[0][:200] + "...")

json_logs wrote 15 lines to /var/folders/lz/f4qx65kx5dn_5mxblzh2s5h00000gn/T/tmpkri346gt.jsonl
first line preview:
{"type": "query", "id": "n_57fd2e9b8682", "branch_id": "main", "children": [], "agent_id": "root", "depth": 0, "query": "Create a simple boids simulation in plain html and javascript. Render 100s of f...
